# Samanantar Average Cosine Similarity Benchmark

This notebook evaluates multilingual / Indic embedding models on **Samanantar** using **average cosine similarity**.

Main question:

> For each English–Indic translation pair, how similar are the two embeddings on average?

This notebook does **not** compute retrieval Accuracy@1.  
It computes pairwise cosine similarity for gold translation pairs and averages them per model and language pair.

Outputs:
- `samanantar_avg_cosine_metrics.csv`
- `model_summary_avg_cosine.csv`
- `pair_summary_avg_cosine.csv`
- per-pair cosine score CSV files
- plots for model-level and language-pair-level comparison


## 0. Colab setup

In Colab, use:

**Runtime → Change runtime type → GPU**

Then run the cells below.


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
# Clean packages that sometimes break text-only Colab runs.
!pip -q uninstall -y torchvision torchaudio torchtext fastai timm -q

# Install only the packages needed for this text embedding benchmark.
# Do not manually install torch; let Colab keep its working GPU torch.
!pip -q install \
  "numpy==2.0.2" \
  "scipy==1.15.3" \
  "scikit-learn==1.6.1" \
  "transformers==4.48.3" \
  "sentence-transformers==3.4.1" \
  "datasets==3.2.0" \
  "accelerate==1.3.0" \
  "pandas==2.2.2" \
  "tqdm==4.67.1" \
  "matplotlib==3.10.0" \
  "pyyaml==6.0.2" \
  "sentencepiece==0.2.0"


In [ ]:
import torch
import numpy as np
import scipy
import sklearn
import transformers
import sentence_transformers
import datasets
import pandas as pd

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)


## 1. Mount Google Drive

This saves samples, embeddings, metrics, and plots permanently.


In [ ]:
from google.colab import drive
from pathlib import Path

USE_DRIVE = True

if USE_DRIVE:
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/indic_embedding_benchmark")
else:
    BASE_DIR = Path("/content/indic_embedding_benchmark")

OUTPUT_DIR = BASE_DIR / "outputs" / "samanantar_avg_cosine"
(OUTPUT_DIR / "samples").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "embeddings").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "scores").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)

print("Saving outputs to:", OUTPUT_DIR)


## 2. Configuration

Samanantar is mainly an **English ↔ Indic parallel corpus**, so this notebook evaluates English–Indic language pairs.

You can start with `MAX_PAIRS_PER_LANGUAGE = 500` for a quick run, then increase it to `2000` or more.


In [ ]:
import gc
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 128
BATCH_SIZE = 32

# Start small; increase later if runtime permits.
MAX_PAIRS_PER_LANGUAGE = 2000

# Samanantar configs are English-Indic.
# lang_pair key is used in output tables.
SAMANANTAR_LANGS = {
    "eng-asm": {"config": "as", "source_language": "eng", "target_language": "asm", "name": "Assamese"},
    "eng-ben": {"config": "bn", "source_language": "eng", "target_language": "ben", "name": "Bengali"},
    "eng-guj": {"config": "gu", "source_language": "eng", "target_language": "guj", "name": "Gujarati"},
    "eng-hin": {"config": "hi", "source_language": "eng", "target_language": "hin", "name": "Hindi"},
    "eng-kan": {"config": "kn", "source_language": "eng", "target_language": "kan", "name": "Kannada"},
    "eng-mal": {"config": "ml", "source_language": "eng", "target_language": "mal", "name": "Malayalam"},
    "eng-mar": {"config": "mr", "source_language": "eng", "target_language": "mar", "name": "Marathi"},
    "eng-ory": {"config": "or", "source_language": "eng", "target_language": "ory", "name": "Odia"},
    "eng-pan": {"config": "pa", "source_language": "eng", "target_language": "pan", "name": "Punjabi"},
    "eng-tam": {"config": "ta", "source_language": "eng", "target_language": "tam", "name": "Tamil"},
    "eng-tel": {"config": "te", "source_language": "eng", "target_language": "tel", "name": "Telugu"},
}

# Add/remove models here.
# sentence_transformer = use SentenceTransformers encode()
# hf_mean_pool = use Hugging Face AutoModel + attention-mask-aware mean pooling
MODELS = [
    {"name": "labse", "hf_id": "sentence-transformers/LaBSE", "kind": "sentence_transformer"},
    {"name": "mpnet_multilingual", "hf_id": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2", "kind": "sentence_transformer"},
    {"name": "sbert_multilingual_minilm", "hf_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "kind": "sentence_transformer"},
    {"name": "muril", "hf_id": "google/muril-base-cased", "kind": "hf_mean_pool"},
    {"name": "indicbertv2_ss", "hf_id": "ai4bharat/IndicBERTv2-SS", "kind": "hf_mean_pool", "trust_remote_code": True},
    {"name": "xlm_roberta_base", "hf_id": "FacebookAI/xlm-roberta-base", "kind": "hf_mean_pool"},
    {"name": "mcontriever", "hf_id": "facebook/mcontriever", "kind": "hf_mean_pool"},
]

RUN_TAG = f"samanantar_n{MAX_PAIRS_PER_LANGUAGE}_l{MAX_LENGTH}"
print("Device:", DEVICE)
print("Run tag:", RUN_TAG)
print("Number of language pairs:", len(SAMANANTAR_LANGS))
print("Number of models:", len(MODELS))


## 3. Load Samanantar samples

This uses streaming so that Colab does not need to download the entire Samanantar dataset.

For each language pair, we save a sampled CSV so reruns are faster and consistent.


In [ ]:
def extract_en_indic_text(row: dict, config_code: str) -> Tuple[str, str]:
    """Robustly extract English and Indic text from a Samanantar row."""

    # Most common Samanantar style
    if "src" in row and "tgt" in row:
        return str(row["src"]), str(row["tgt"])

    # Alternative naming
    if "source" in row and "target" in row:
        return str(row["source"]), str(row["target"])

    # Sometimes translation datasets store language columns directly
    if "en" in row and config_code in row:
        return str(row["en"]), str(row[config_code])

    if "eng" in row and config_code in row:
        return str(row["eng"]), str(row[config_code])

    # Helpful debug if dataset format changes
    raise KeyError(f"Could not find text fields in row. Available keys: {list(row.keys())}")


def load_samanantar_pairs(lang_pair: str, info: dict, max_pairs: int) -> pd.DataFrame:
    """Load or sample English-Indic pairs for one Samanantar config."""

    sample_path = OUTPUT_DIR / "samples" / f"{lang_pair}_n{max_pairs}.csv"

    if sample_path.exists():
        df = pd.read_csv(sample_path)
        print(f"Loaded cached sample: {sample_path.name} | rows={len(df)}")
        return df

    config = info["config"]
    print(f"Streaming Samanantar config={config} for {lang_pair}")

    ds = load_dataset(
        "ai4bharat/samanantar",
        config,
        split="train",
        streaming=True,
    )

    # Shuffle with a buffer so we do not always take the same first rows.
    ds = ds.shuffle(seed=SEED, buffer_size=10_000)

    rows = []
    for row in tqdm(ds, total=max_pairs, desc=f"Sampling {lang_pair}"):
        try:
            en_text, indic_text = extract_en_indic_text(row, config)
        except Exception as e:
            print("Row extraction failed. Example row:", row)
            raise e

        en_text = en_text.strip()
        indic_text = indic_text.strip()

        if not en_text or not indic_text:
            continue

        rows.append({
            "language_pair": lang_pair,
            "source_language": info["source_language"],
            "target_language": info["target_language"],
            "source_text": en_text,
            "target_text": indic_text,
        })

        if len(rows) >= max_pairs:
            break

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(f"No rows loaded for {lang_pair}. Check dataset fields/config.")

    df.to_csv(sample_path, index=False)
    print(f"Saved sample: {sample_path.name} | rows={len(df)}")
    return df


pairs_by_lang = {}

for lang_pair, info in SAMANANTAR_LANGS.items():
    pairs_by_lang[lang_pair] = load_samanantar_pairs(
        lang_pair,
        info,
        MAX_PAIRS_PER_LANGUAGE,
    )

print("\nLoaded language pairs:")
for lang_pair, df in pairs_by_lang.items():
    print(lang_pair, df.shape)

# Preview one pair
example_pair = list(pairs_by_lang.keys())[0]
display(pairs_by_lang[example_pair].head())


## 4. Embedding helpers

For sentence-transformer models, we use the model's own `encode()` function.

For raw Hugging Face models, we use attention-mask-aware mean pooling.


In [ ]:
@dataclass
class ModelSpec:
    name: str
    hf_id: str
    kind: str
    trust_remote_code: bool = False


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


class Embedder:
    def __init__(self, spec: ModelSpec, device: str, max_length: int):
        self.spec = spec
        self.device = device
        self.max_length = max_length

        if spec.kind == "sentence_transformer":
            from sentence_transformers import SentenceTransformer

            print("Loading as SentenceTransformer:", spec.hf_id)
            self.model = SentenceTransformer(
                spec.hf_id,
                device=device,
                trust_remote_code=spec.trust_remote_code,
            )

            # Save GPU memory on Colab
            if device == "cuda":
                self.model = self.model.half()

            self.tokenizer = None

        elif spec.kind == "hf_mean_pool":
            from transformers import AutoModel, AutoTokenizer

            print("Loading as HF mean-pool model:", spec.hf_id)

            self.tokenizer = AutoTokenizer.from_pretrained(
                spec.hf_id,
                trust_remote_code=spec.trust_remote_code,
            )

            dtype = torch.float16 if device == "cuda" else torch.float32

            self.model = AutoModel.from_pretrained(
                spec.hf_id,
                trust_remote_code=spec.trust_remote_code,
                torch_dtype=dtype,
            )

            self.model.to(device)
            self.model.eval()

        else:
            raise ValueError(f"Unknown model kind: {spec.kind}")

    @torch.no_grad()
    def encode(self, texts: List[str], batch_size: int) -> np.ndarray:
        if self.spec.kind == "sentence_transformer":
            emb = self.model.encode(
                texts,
                batch_size=batch_size,
                convert_to_numpy=True,
                normalize_embeddings=False,  # cosine_similarity will normalize internally
                show_progress_bar=True,
            )
            return emb.astype("float32")

        all_embeddings = []

        for start in tqdm(range(0, len(texts), batch_size), desc=f"Encoding {self.spec.name}"):
            batch = texts[start:start + batch_size]

            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )

            encoded = {k: v.to(self.device) for k, v in encoded.items()}
            outputs = self.model(**encoded)

            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            all_embeddings.append(pooled.detach().cpu().float().numpy())

        return np.vstack(all_embeddings).astype("float32")


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 5. Cosine similarity metrics

This benchmark uses **explicit cosine similarity** from scikit-learn.

For each gold translation pair:
- English sentence embedding
- Indic translation embedding

we compute:

```python
cosine_similarity(english_embedding, indic_embedding)
```

Then we average the scores per model and language pair.


In [ ]:
def paired_cosine_scores(
    src_emb: np.ndarray,
    tgt_emb: np.ndarray,
    batch_size: int = 256
) -> np.ndarray:
    """Compute cosine similarity only for aligned gold pairs.

    src_emb[i] and tgt_emb[i] are translations of each other.
    This avoids creating a huge all-vs-all matrix.
    """

    if src_emb.shape[0] != tgt_emb.shape[0]:
        raise ValueError(f"Length mismatch: {src_emb.shape[0]} vs {tgt_emb.shape[0]}")

    scores = []

    for start in range(0, src_emb.shape[0], batch_size):
        end = start + batch_size

        # cosine_similarity creates a batch_size x batch_size matrix.
        # The diagonal contains the aligned pair scores.
        sim_batch = cosine_similarity(src_emb[start:end], tgt_emb[start:end])
        diag_scores = np.diag(sim_batch)

        scores.append(diag_scores.astype("float32"))

    return np.concatenate(scores)


def summarize_cosine_scores(scores: np.ndarray) -> Dict[str, float]:
    return {
        "mean_cosine": float(np.mean(scores)),
        "std_cosine": float(np.std(scores)),
        "median_cosine": float(np.median(scores)),
        "min_cosine": float(np.min(scores)),
        "max_cosine": float(np.max(scores)),
        "p05_cosine": float(np.percentile(scores, 5)),
        "p25_cosine": float(np.percentile(scores, 25)),
        "p75_cosine": float(np.percentile(scores, 75)),
        "p95_cosine": float(np.percentile(scores, 95)),
        "pct_gt_0_50": float(np.mean(scores > 0.50)),
        "pct_gt_0_60": float(np.mean(scores > 0.60)),
        "pct_gt_0_70": float(np.mean(scores > 0.70)),
        "pct_gt_0_80": float(np.mean(scores > 0.80)),
    }


## 6. Run benchmark

This loop:

1. loads each model,
2. encodes source and target sentences for each language pair,
3. computes cosine scores for aligned translation pairs,
4. averages the cosine scores,
5. saves results.


In [ ]:
all_rows = []

for model_dict in MODELS:
    spec = ModelSpec(**model_dict)
    print(f"\n===== Model: {spec.name} | {spec.hf_id} =====")

    embedder = None

    try:
        embedder = Embedder(spec, DEVICE, MAX_LENGTH)

        for lang_pair, df in pairs_by_lang.items():
            src_texts = df["source_text"].astype(str).tolist()
            tgt_texts = df["target_text"].astype(str).tolist()

            src_lang = df["source_language"].iloc[0]
            tgt_lang = df["target_language"].iloc[0]

            src_cache_path = (
                OUTPUT_DIR
                / "embeddings"
                / f"{spec.name}_{lang_pair}_SRC_{RUN_TAG}.npy"
            )
            tgt_cache_path = (
                OUTPUT_DIR
                / "embeddings"
                / f"{spec.name}_{lang_pair}_TGT_{RUN_TAG}.npy"
            )

            # Encode / load source embeddings
            if src_cache_path.exists():
                src_emb = np.load(src_cache_path)
                print("Loaded cached source:", src_cache_path.name, src_emb.shape)
            else:
                src_emb = embedder.encode(src_texts, batch_size=BATCH_SIZE)
                np.save(src_cache_path, src_emb)
                print("Saved source:", src_cache_path.name, src_emb.shape)

            # Encode / load target embeddings
            if tgt_cache_path.exists():
                tgt_emb = np.load(tgt_cache_path)
                print("Loaded cached target:", tgt_cache_path.name, tgt_emb.shape)
            else:
                tgt_emb = embedder.encode(tgt_texts, batch_size=BATCH_SIZE)
                np.save(tgt_cache_path, tgt_emb)
                print("Saved target:", tgt_cache_path.name, tgt_emb.shape)

            # Safety check
            if src_emb.shape[0] != len(src_texts) or tgt_emb.shape[0] != len(tgt_texts):
                print("Cache size mismatch. Recomputing both source and target embeddings...")
                src_emb = embedder.encode(src_texts, batch_size=BATCH_SIZE)
                tgt_emb = embedder.encode(tgt_texts, batch_size=BATCH_SIZE)
                np.save(src_cache_path, src_emb)
                np.save(tgt_cache_path, tgt_emb)

            # Explicit cosine similarity for aligned gold translation pairs
            scores = paired_cosine_scores(src_emb, tgt_emb, batch_size=256)

            score_df = pd.DataFrame({
                "language_pair": lang_pair,
                "source_language": src_lang,
                "target_language": tgt_lang,
                "cosine_similarity": scores,
                "source_text": src_texts,
                "target_text": tgt_texts,
            })

            score_path = OUTPUT_DIR / "scores" / f"{spec.name}_{lang_pair}_{RUN_TAG}_cosine_scores.csv"
            score_df.to_csv(score_path, index=False)

            stats = summarize_cosine_scores(scores)

            row = {
                "model": spec.name,
                "hf_id": spec.hf_id,
                "language_pair": lang_pair,
                "source_language": src_lang,
                "target_language": tgt_lang,
                "n_pairs": len(scores),
                **stats,
            }

            all_rows.append(row)

            print(
                lang_pair,
                "Mean cosine:", round(row["mean_cosine"], 4),
                "Median:", round(row["median_cosine"], 4),
                "% > 0.80:", round(row["pct_gt_0_80"] * 100, 2)
            )

    except Exception as e:
        print("FAILED model:", spec.name)
        print(type(e).__name__, ":", e)
        print("Skipping this model and moving to the next one.")

    finally:
        if embedder is not None:
            del embedder
        clear_memory()


metrics_df = pd.DataFrame(all_rows)

if metrics_df.empty:
    raise ValueError("No model produced results. Check model loading and dataset loading errors above.")

metrics_path = OUTPUT_DIR / f"samanantar_avg_cosine_metrics_{RUN_TAG}.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\nSaved metrics:", metrics_path)
print("metrics_df shape:", metrics_df.shape)
display(metrics_df.head())


## 7. Model-level summary

This averages each model across all Samanantar language pairs.


In [ ]:
model_summary = (
    metrics_df
    .groupby("model")[[
        "mean_cosine",
        "median_cosine",
        "pct_gt_0_60",
        "pct_gt_0_70",
        "pct_gt_0_80",
    ]]
    .mean()
    .sort_values("mean_cosine", ascending=False)
)

model_summary_path = OUTPUT_DIR / f"model_summary_avg_cosine_{RUN_TAG}.csv"
model_summary.to_csv(model_summary_path)

print("Saved model summary:", model_summary_path)
display(model_summary)


## 8. Language-pair summary

This keeps model and language pair separate.


In [ ]:
pair_summary = (
    metrics_df
    .sort_values(["model", "mean_cosine"], ascending=[True, False])
    .reset_index(drop=True)
)

pair_summary_path = OUTPUT_DIR / f"pair_summary_avg_cosine_{RUN_TAG}.csv"
pair_summary.to_csv(pair_summary_path, index=False)

print("Saved pair summary:", pair_summary_path)
display(pair_summary.head(30))


## 9. Plots


In [ ]:
# Average mean cosine by model
model_avg = model_summary["mean_cosine"].sort_values(ascending=False)

ax = model_avg.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Average Mean Cosine Similarity")
ax.set_title("Samanantar: Average Gold-Pair Cosine Similarity by Model")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plot_path = OUTPUT_DIR / "plots" / f"avg_mean_cosine_by_model_{RUN_TAG}.png"
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved plot:", plot_path)


In [ ]:
# Model x language-pair mean cosine chart
pivot = metrics_df.pivot(
    index="model",
    columns="language_pair",
    values="mean_cosine"
)

ax = pivot.plot(kind="bar", figsize=(16, 6))
ax.set_ylabel("Mean Cosine Similarity")
ax.set_title("Samanantar: Mean Gold-Pair Cosine Similarity by Model and Language Pair")
ax.legend(title="Language pair", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plot_path = OUTPUT_DIR / "plots" / f"mean_cosine_by_model_and_pair_{RUN_TAG}.png"
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved plot:", plot_path)


## 10. Best and weakest language pairs per model


In [ ]:
best_pairs = (
    metrics_df
    .sort_values(["model", "mean_cosine"], ascending=[True, False])
    .groupby("model")
    .head(5)
)

weakest_pairs = (
    metrics_df
    .sort_values(["model", "mean_cosine"], ascending=[True, True])
    .groupby("model")
    .head(5)
)

best_path = OUTPUT_DIR / f"best_pairs_avg_cosine_{RUN_TAG}.csv"
weakest_path = OUTPUT_DIR / f"weakest_pairs_avg_cosine_{RUN_TAG}.csv"

best_pairs.to_csv(best_path, index=False)
weakest_pairs.to_csv(weakest_path, index=False)

print("Best pairs:")
display(best_pairs[["model", "language_pair", "mean_cosine", "median_cosine", "pct_gt_0_80"]])

print("Weakest pairs:")
display(weakest_pairs[["model", "language_pair", "mean_cosine", "median_cosine", "pct_gt_0_80"]])

print("Saved:", best_path)
print("Saved:", weakest_path)


## 11. Inspect pair-level cosine scores

Use this to see individual sentence-pair scores for one model/language pair.


In [ ]:
import glob

score_files = sorted(glob.glob(str(OUTPUT_DIR / "scores" / f"*_{RUN_TAG}_cosine_scores.csv")))

print("Number of score files:", len(score_files))
print("\nFirst 10 score files:")
for p in score_files[:10]:
    print(p)

SCORE_FILE_INDEX = 0

if score_files:
    sample_path = score_files[SCORE_FILE_INDEX]
    print("\nShowing:", sample_path)
    sample_scores = pd.read_csv(sample_path)
    display(sample_scores.head(10))


## 12. Interpretation guide

Use these in the report:

- **Mean cosine**: average similarity of correct translation pairs.
- **Median cosine**: typical similarity score.
- **% > 0.80**: fraction of gold pairs that cross a strict similarity threshold.
- This benchmark is different from retrieval Accuracy@1.  
  A pair may have cosine < 0.80 but still be the nearest neighbour in a retrieval benchmark.
